# EuroSAT-SAR Classification and Image Retrieval

Final notebook for dual-polarization SAR training, complete test-set retrieval
evaluation, artifact export, and Hugging Face deployment.

The original SAR TIFF contains two channels. The notebook converts them into a
three-channel representation suitable for an ImageNet backbone:

- VV
- VH
- Mean of VV and VH


In [ ]:
import json
import random
import re
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import tifffile
from PIL import Image
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

ROOT_DIR = Path("/kaggle/input/datasets/glitchr/eurodata")
SAR_DIR = ROOT_DIR / "EuroSAT-SAR"
WORK_DIR = Path("/kaggle/working/eurodata_sar")
MODEL_DIR = WORK_DIR / "models"
REPORT_DIR = WORK_DIR / "reports"
DEPLOY_DIR = WORK_DIR / "huggingface_space"

for directory in (WORK_DIR, MODEL_DIR, REPORT_DIR, DEPLOY_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "DenseNet121_SAR"
IMAGE_SIZE = 64
INPUT_CHANNELS = 2
MODEL_CHANNELS = 3
INPUT_SHAPE = (IMAGE_SIZE, IMAGE_SIZE, MODEL_CHANNELS)
BATCH_SIZE = 64
EPOCHS = 15
LEARNING_RATE = 1e-3
AUTOTUNE = tf.data.AUTOTUNE

CLASS_NAMES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
    "Pasture", "PermanentCrop", "Residential", "River", "SeaLake",
]
CLASS_TO_INDEX = {
    label: index for index, label in enumerate(CLASS_NAMES)
}
NUM_CLASSES = len(CLASS_NAMES)
VALID_EXTENSIONS = {".tif", ".tiff"}

print("TensorFlow:", tf.__version__)
print("Model:", MODEL_NAME)
print("SAR directory:", SAR_DIR)


## 1. Build and validate the SAR dataset index

In [ ]:
def extract_file_id(image_path):
    numbers = re.findall(r"\d+", image_path.stem)
    return int(numbers[-1]) if numbers else None


def collect_sar_files(root_dir):
    records = []

    for label in CLASS_NAMES:
        class_dir = root_dir / label

        if not class_dir.exists():
            raise FileNotFoundError(f"Missing class directory: {class_dir}")

        for image_path in sorted(class_dir.rglob("*")):
            if (
                image_path.is_file()
                and image_path.suffix.lower() in VALID_EXTENSIONS
            ):
                records.append(
                    {
                        "file_id": extract_file_id(image_path),
                        "label": label,
                        "target": CLASS_TO_INDEX[label],
                        "image_path": str(image_path),
                    }
                )

    dataframe = pd.DataFrame(records)
    dataframe = dataframe.dropna(
        subset=["file_id", "label", "image_path"]
    )
    dataframe["file_id"] = dataframe["file_id"].astype(int)

    duplicate_count = int(
        dataframe.duplicated(["label", "file_id"]).sum()
    )

    dataframe = dataframe.drop_duplicates(
        subset=["label", "file_id"],
        keep="first",
    ).reset_index(drop=True)

    print("Rows after deduplication:", len(dataframe))
    print("Duplicates removed:", duplicate_count)
    return dataframe


final_df = collect_sar_files(SAR_DIR)

assert final_df["label"].isin(CLASS_NAMES).all()
assert final_df["target"].between(0, NUM_CLASSES - 1).all()
assert not final_df.duplicated(["label", "file_id"]).any()

display(final_df.head())
display(final_df["label"].value_counts().sort_index())


In [ ]:
train_df, temporary_df = train_test_split(
    final_df,
    test_size=0.30,
    random_state=SEED,
    stratify=final_df["label"],
)

val_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temporary_df["label"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

display(
    pd.DataFrame(
        {
            "split": ["train", "validation", "test"],
            "images": [len(train_df), len(val_df), len(test_df)],
        }
    )
)


## 2. Inspect SAR channel structure

In [ ]:
sample_path = Path(final_df.iloc[0]["image_path"])
sample_array = tifffile.imread(sample_path)

print("Sample path:", sample_path)
print("Original shape:", sample_array.shape)
print("Data type:", sample_array.dtype)
print("Minimum:", np.nanmin(sample_array))
print("Maximum:", np.nanmax(sample_array))


## 3. SAR preprocessing and TensorFlow pipeline

In [ ]:
def percentile_stretch_numpy(channel):
    channel = np.asarray(channel, dtype=np.float32)
    low = np.percentile(channel, 2)
    high = np.percentile(channel, 98)

    if high <= low:
        return np.zeros_like(channel, dtype=np.float32)

    channel = (channel - low) / (high - low)
    return np.clip(channel, 0.0, 1.0).astype(np.float32)


def load_sar_numpy(path_bytes):
    path = path_bytes.decode("utf-8")
    array = tifffile.imread(path)
    array = np.asarray(array, dtype=np.float32)

    if array.ndim == 3 and array.shape[0] == INPUT_CHANNELS:
        array = np.moveaxis(array, 0, -1)

    if array.ndim == 2:
        array = np.stack([array, array], axis=-1)

    if array.ndim != 3 or array.shape[-1] < INPUT_CHANNELS:
        raise ValueError(
            f"Expected two SAR channels, received shape {array.shape}"
        )

    vv = percentile_stretch_numpy(array[..., 0])
    vh = percentile_stretch_numpy(array[..., 1])
    mean_channel = (vv + vh) / 2.0

    image = np.stack([vv, vh, mean_channel], axis=-1)
    return (image * 255.0).astype(np.float32)


def decode_sar(image_path, target):
    image = tf.numpy_function(
        load_sar_numpy,
        [image_path],
        Tout=tf.float32,
    )
    image.set_shape([None, None, MODEL_CHANNELS])
    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    target = tf.one_hot(target, depth=NUM_CLASSES)
    return image, target


def augment_sar(image, target):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=8.0)
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, target


def create_dataset(dataframe, training=False):
    paths = dataframe["image_path"].to_numpy()
    targets = dataframe["target"].to_numpy(dtype=np.int32)

    dataset = tf.data.Dataset.from_tensor_slices((paths, targets))

    if training:
        dataset = dataset.shuffle(
            len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    dataset = dataset.map(
        decode_sar,
        num_parallel_calls=AUTOTUNE,
    )

    if training:
        dataset = dataset.map(
            augment_sar,
            num_parallel_calls=AUTOTUNE,
        )

    return dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = create_dataset(train_df, training=True)
val_ds = create_dataset(val_df)
test_ds = create_dataset(test_df)


In [ ]:
sample_images, sample_targets = next(iter(train_ds))

print("Batch image shape:", sample_images.shape)
print("Batch target shape:", sample_targets.shape)
print("Image minimum:", tf.reduce_min(sample_images).numpy())
print("Image maximum:", tf.reduce_max(sample_images).numpy())

plt.figure(figsize=(12, 6))

for index in range(8):
    plt.subplot(2, 4, index + 1)
    plt.imshow(sample_images[index].numpy().astype(np.uint8))
    label_index = int(tf.argmax(sample_targets[index]).numpy())
    plt.title(CLASS_NAMES[label_index])
    plt.axis("off")

plt.tight_layout()
plt.show()


## 4. Build and train the SAR model

In [ ]:
MODEL_REGISTRY = {
    "DenseNet121_SAR": tf.keras.applications.DenseNet121,
    "ResNet50_SAR": tf.keras.applications.ResNet50,
    "MobileNetV2_SAR": tf.keras.applications.MobileNetV2,
    "EfficientNetB0_SAR": tf.keras.applications.EfficientNetB0,
}

PREPROCESS_REGISTRY = {
    "DenseNet121_SAR": tf.keras.applications.densenet.preprocess_input,
    "ResNet50_SAR": tf.keras.applications.resnet.preprocess_input,
    "MobileNetV2_SAR": tf.keras.applications.mobilenet_v2.preprocess_input,
    "EfficientNetB0_SAR": tf.keras.applications.efficientnet.preprocess_input,
}


def build_model(model_name):
    if model_name not in MODEL_REGISTRY:
        raise ValueError(
            f"Unsupported model: {model_name}. "
            f"Choose from {list(MODEL_REGISTRY)}"
        )

    inputs = tf.keras.Input(
        shape=INPUT_SHAPE,
        name="sar_image",
    )
    x = tf.keras.layers.Lambda(
        PREPROCESS_REGISTRY[model_name],
        name="model_preprocessing",
    )(inputs)

    backbone = MODEL_REGISTRY[model_name](
        include_top=False,
        weights="imagenet",
        input_shape=INPUT_SHAPE,
        pooling="avg",
    )
    backbone.trainable = False

    x = backbone(x, training=False)
    x = tf.keras.layers.BatchNormalization(
        name="embedding_batch_norm"
    )(x)
    x = tf.keras.layers.Dense(
        256,
        activation="relu",
        name="embedding",
    )(x)
    x = tf.keras.layers.Dropout(0.30)(x)

    outputs = tf.keras.layers.Dense(
        NUM_CLASSES,
        activation="softmax",
        name="classification",
    )(x)

    return tf.keras.Model(
        inputs,
        outputs,
        name=f"{model_name}_EuroSAT_SAR",
    )


model = build_model(MODEL_NAME)
model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.TopKCategoricalAccuracy(
            k=5,
            name="top_5_accuracy",
        ),
    ],
)
model.summary()


In [ ]:
CLASSIFIER_PATH = MODEL_DIR / f"{MODEL_NAME}_classifier.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        CLASSIFIER_PATH,
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
]

training_start = time.perf_counter()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

training_seconds = time.perf_counter() - training_start

model = tf.keras.models.load_model(
    CLASSIFIER_PATH,
    compile=False,
    safe_mode=False,
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.TopKCategoricalAccuracy(
            k=5,
            name="top_5_accuracy",
        ),
    ],
)

test_metrics = model.evaluate(
    test_ds,
    return_dict=True,
    verbose=1,
)
test_metrics["training_seconds"] = training_seconds

display(pd.DataFrame([test_metrics]))


## 5. Classification validation

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.to_csv(REPORT_DIR / "training_history.csv", index=False)

plt.figure(figsize=(8, 5))
plt.plot(history_df["accuracy"], label="Training")
plt.plot(history_df["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title(f"{MODEL_NAME} accuracy")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="Training")
plt.plot(history_df["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"{MODEL_NAME} loss")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


In [ ]:
probabilities = model.predict(test_ds, verbose=1)
predicted_classes = probabilities.argmax(axis=1)
true_classes = test_df["target"].to_numpy()

report = classification_report(
    true_classes,
    predicted_classes,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(
    REPORT_DIR / "classification_report.csv",
    index=True,
)
display(report_df)

matrix = confusion_matrix(true_classes, predicted_classes)

ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=CLASS_NAMES,
).plot(xticks_rotation=45)

plt.title(f"{MODEL_NAME} confusion matrix")
plt.tight_layout()
plt.show()


## 6. Complete test-set SAR retrieval evaluation

In [ ]:
embedding_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("embedding").output,
    name=f"{MODEL_NAME}_embedding_model",
)

EMBEDDING_MODEL_PATH = (
    MODEL_DIR / f"{MODEL_NAME}_embedding.keras"
)
embedding_model.save(EMBEDDING_MODEL_PATH)

gallery_embeddings = embedding_model.predict(
    test_ds,
    verbose=1,
).astype(np.float32)

gallery_embeddings /= np.clip(
    np.linalg.norm(gallery_embeddings, axis=1, keepdims=True),
    1e-8,
    None,
)

assert len(gallery_embeddings) == len(test_df)
assert np.isfinite(gallery_embeddings).all()
assert np.allclose(
    np.linalg.norm(gallery_embeddings, axis=1),
    1.0,
    atol=1e-5,
)

print("Embedding shape:", gallery_embeddings.shape)


In [ ]:
def evaluate_retrieval(
    dataframe,
    embeddings,
    k_values=(1, 5, 10, 20, 50),
):
    labels = dataframe["label"].to_numpy()
    embeddings = np.asarray(embeddings, dtype=np.float32)

    if len(dataframe) != len(embeddings):
        raise ValueError(
            "Dataframe and embedding counts do not match."
        )

    gallery_size = len(dataframe) - 1
    valid_k_values = sorted(
        {
            min(int(k), gallery_size)
            for k in k_values
            if int(k) > 0
        }
    )

    query_rows = []
    retrieval_times = []

    for query_index in tqdm(
        range(len(dataframe)),
        desc="Evaluating all SAR test queries",
    ):
        start = time.perf_counter()

        similarities = embeddings @ embeddings[query_index]
        similarities[query_index] = -np.inf
        ranked_indices = np.argsort(similarities)[::-1]

        retrieval_times.append(time.perf_counter() - start)

        query_label = labels[query_index]
        relevance = (
            labels[ranked_indices] == query_label
        ).astype(np.int32)

        total_relevant = int(
            np.sum(labels == query_label) - 1
        )
        cumulative_relevant = np.cumsum(relevance)

        row = {
            "query_index": query_index,
            "query_label": query_label,
            "total_relevant": total_relevant,
        }

        for k in valid_k_values:
            relevant_at_k = int(cumulative_relevant[k - 1])
            precision = relevant_at_k / k
            recall = (
                relevant_at_k / total_relevant
                if total_relevant > 0
                else 0.0
            )
            f1 = (
                2 * precision * recall / (precision + recall)
                if precision + recall > 0
                else 0.0
            )

            row[f"relevant_at_{k}"] = relevant_at_k
            row[f"precision_at_{k}"] = precision
            row[f"recall_at_{k}"] = recall
            row[f"f1_at_{k}"] = f1

        ranks = np.arange(1, len(relevance) + 1)
        precision_by_rank = cumulative_relevant / ranks

        average_precision = (
            np.sum(precision_by_rank * relevance) / total_relevant
            if total_relevant > 0
            else 0.0
        )

        relevant_positions = np.flatnonzero(relevance)
        reciprocal_rank = (
            1.0 / (relevant_positions[0] + 1)
            if len(relevant_positions)
            else 0.0
        )

        row["average_precision"] = float(average_precision)
        row["reciprocal_rank"] = float(reciprocal_rank)
        query_rows.append(row)

    query_metrics = pd.DataFrame(query_rows)

    summary = {
        "model_name": MODEL_NAME,
        "modality": "SAR",
        "number_of_queries": len(dataframe),
        "gallery_size_per_query": gallery_size,
        "mean_average_precision": (
            query_metrics["average_precision"].mean()
        ),
        "mean_reciprocal_rank": (
            query_metrics["reciprocal_rank"].mean()
        ),
        "average_retrieval_time_ms": (
            np.mean(retrieval_times) * 1000
        ),
        "queries_per_second": (
            1.0 / np.mean(retrieval_times)
        ),
    }

    for k in valid_k_values:
        for metric in ("precision", "recall", "f1"):
            column = f"{metric}_at_{k}"
            summary[column] = query_metrics[column].mean()

    summary_df = pd.DataFrame([summary])

    metric_columns = [
        column
        for column in query_metrics.columns
        if column.startswith(("precision_at_", "recall_at_", "f1_at_"))
    ]
    metric_values = query_metrics[metric_columns].to_numpy()

    assert np.isfinite(metric_values).all()
    assert ((metric_values >= 0) & (metric_values <= 1)).all()

    return query_metrics, summary_df, valid_k_values


In [ ]:
K_VALUES = [1, 2, 3, 5, 10, 15, 20, 30, 50]

query_metrics_df, retrieval_summary_df, valid_k_values = (
    evaluate_retrieval(
        dataframe=test_df,
        embeddings=gallery_embeddings,
        k_values=K_VALUES,
    )
)

query_metrics_df.to_csv(
    REPORT_DIR / "query_retrieval_metrics.csv",
    index=False,
)
retrieval_summary_df.to_csv(
    REPORT_DIR / "retrieval_summary.csv",
    index=False,
)

display(retrieval_summary_df.T)


In [ ]:
precision_values = [
    retrieval_summary_df[f"precision_at_{k}"].iloc[0]
    for k in valid_k_values
]
recall_values = [
    retrieval_summary_df[f"recall_at_{k}"].iloc[0]
    for k in valid_k_values
]
f1_values = [
    retrieval_summary_df[f"f1_at_{k}"].iloc[0]
    for k in valid_k_values
]

plt.figure(figsize=(9, 6))
plt.plot(
    valid_k_values,
    precision_values,
    marker="o",
    label="Mean Precision@K",
)
plt.plot(
    valid_k_values,
    recall_values,
    marker="x",
    label="Mean Recall@K",
)
plt.plot(
    valid_k_values,
    f1_values,
    marker="*",
    label="Mean F1@K",
)

plt.xlabel("Top-K retrieved SAR images")
plt.ylabel("Mean score")
plt.title(f"{MODEL_NAME}: complete SAR test-set retrieval")
plt.xticks(valid_k_values)
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()


In [ ]:
class_metric_columns = [
    "precision_at_5",
    "recall_at_5",
    "f1_at_5",
    "precision_at_10",
    "recall_at_10",
    "f1_at_10",
]

class_metrics_df = (
    query_metrics_df.groupby("query_label")[class_metric_columns]
    .mean()
    .reset_index()
)

class_metrics_df.to_csv(
    REPORT_DIR / "class_retrieval_metrics.csv",
    index=False,
)

display(class_metrics_df.round(4))


## 7. Export Hugging Face deployment files

In [ ]:
def sar_preview(image_path):
    array = tifffile.imread(image_path)
    array = np.asarray(array, dtype=np.float32)

    if array.ndim == 3 and array.shape[0] == INPUT_CHANNELS:
        array = np.moveaxis(array, 0, -1)

    if array.ndim == 2:
        array = np.stack([array, array], axis=-1)

    vv = percentile_stretch_numpy(array[..., 0])
    vh = percentile_stretch_numpy(array[..., 1])
    mean_channel = (vv + vh) / 2.0

    preview = np.stack([vv, vh, mean_channel], axis=-1)
    return (preview * 255.0).astype(np.uint8)


CLASSIFIER_DEPLOY_PATH = (
    DEPLOY_DIR / f"{MODEL_NAME}_classifier.keras"
)
EMBEDDING_DEPLOY_PATH = (
    DEPLOY_DIR / f"{MODEL_NAME}_embedding.keras"
)
EMBEDDINGS_DEPLOY_PATH = (
    DEPLOY_DIR / f"{MODEL_NAME}_gallery_embeddings.npy"
)
METADATA_DEPLOY_PATH = (
    DEPLOY_DIR / f"{MODEL_NAME}_gallery_metadata.csv"
)
PREVIEW_DIR = DEPLOY_DIR / "gallery_previews"
PREVIEW_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(CLASSIFIER_PATH, CLASSIFIER_DEPLOY_PATH)
shutil.copy2(EMBEDDING_MODEL_PATH, EMBEDDING_DEPLOY_PATH)
np.save(EMBEDDINGS_DEPLOY_PATH, gallery_embeddings)

deployment_metadata = test_df[
    ["file_id", "label", "image_path"]
].copy()

relative_preview_paths = []

for row in tqdm(
    deployment_metadata.itertuples(index=False),
    total=len(deployment_metadata),
    desc="Creating SAR previews",
):
    class_dir = PREVIEW_DIR / row.label
    class_dir.mkdir(parents=True, exist_ok=True)

    preview_name = f"{row.label}_{row.file_id}.png"
    preview_path = class_dir / preview_name

    if not preview_path.exists():
        preview = sar_preview(row.image_path)
        Image.fromarray(preview).save(preview_path)

    relative_preview_paths.append(
        preview_path.relative_to(DEPLOY_DIR).as_posix()
    )

deployment_metadata["relative_preview_path"] = (
    relative_preview_paths
)
deployment_metadata = deployment_metadata.drop(
    columns=["image_path"]
)
deployment_metadata.to_csv(
    METADATA_DEPLOY_PATH,
    index=False,
)


In [ ]:
APP_CODE = 'import json\nfrom pathlib import Path\n\nimport gradio as gr\nimport numpy as np\nimport pandas as pd\nimport tensorflow as tf\nimport tifffile\nfrom PIL import Image\n\nROOT = Path(__file__).resolve().parent\n\nwith open(ROOT / "config.json", "r", encoding="utf-8") as file:\n    CONFIG = json.load(file)\n\nMODEL_NAME = CONFIG["model_name"]\nIMAGE_SIZE = int(CONFIG["image_size"])\nCLASS_NAMES = CONFIG["class_names"]\nTOP_K_DEFAULT = int(CONFIG.get("top_k_default", 5))\n\nclassifier = tf.keras.models.load_model(\n    ROOT / CONFIG["model_file"],\n    compile=False,\n    safe_mode=False,\n)\nembedding_model = tf.keras.models.load_model(\n    ROOT / CONFIG["embedding_model_file"],\n    compile=False,\n    safe_mode=False,\n)\n\ngallery_embeddings = np.load(\n    ROOT / CONFIG["gallery_embeddings_file"]\n).astype(np.float32)\ngallery_metadata = pd.read_csv(\n    ROOT / CONFIG["gallery_metadata_file"]\n)\n\ngallery_embeddings /= np.clip(\n    np.linalg.norm(gallery_embeddings, axis=1, keepdims=True),\n    1e-8,\n    None,\n)\n\n\ndef percentile_stretch(array):\n    low = np.percentile(array, 2)\n    high = np.percentile(array, 98)\n\n    if high <= low:\n        return np.zeros_like(array, dtype=np.float32)\n\n    stretched = (array - low) / (high - low)\n    return np.clip(stretched, 0.0, 1.0).astype(np.float32)\n\n\ndef read_sar_image(file_obj):\n    file_bytes = file_obj.read()\n    temporary_path = ROOT / "_temporary_query.tif"\n    temporary_path.write_bytes(file_bytes)\n\n    array = tifffile.imread(temporary_path)\n    temporary_path.unlink(missing_ok=True)\n\n    array = np.asarray(array, dtype=np.float32)\n\n    if array.ndim == 3 and array.shape[0] == 2:\n        array = np.moveaxis(array, 0, -1)\n\n    if array.ndim == 2:\n        array = np.stack([array, array], axis=-1)\n\n    if array.shape[-1] < 2:\n        raise ValueError("SAR image must contain at least two channels.")\n\n    vv = percentile_stretch(array[..., 0])\n    vh = percentile_stretch(array[..., 1])\n    mean_channel = (vv + vh) / 2.0\n\n    image = np.stack([vv, vh, mean_channel], axis=-1)\n    image = tf.image.resize(image, (IMAGE_SIZE, IMAGE_SIZE)).numpy()\n    image = image * 255.0\n\n    preview = Image.fromarray(image.astype(np.uint8))\n    batch = np.expand_dims(image.astype(np.float32), axis=0)\n    return preview, batch\n\n\ndef predict_class(batch):\n    probabilities = classifier.predict(batch, verbose=0)[0]\n    top_indices = np.argsort(probabilities)[::-1][:5]\n\n    return {\n        CLASS_NAMES[index]: float(probabilities[index])\n        for index in top_indices\n    }\n\n\ndef retrieve_images(batch, top_k):\n    query_embedding = embedding_model.predict(batch, verbose=0)\n    query_embedding /= np.clip(\n        np.linalg.norm(query_embedding, axis=1, keepdims=True),\n        1e-8,\n        None,\n    )\n\n    similarities = gallery_embeddings @ query_embedding[0]\n    top_k = min(int(top_k), len(gallery_metadata))\n    ranked_indices = np.argsort(similarities)[::-1][:top_k]\n\n    gallery = []\n    rows = []\n\n    for rank, index in enumerate(ranked_indices, start=1):\n        row = gallery_metadata.iloc[index]\n        preview_path = ROOT / row["relative_preview_path"]\n\n        if preview_path.exists():\n            caption = (\n                f"Rank {rank} | {row[\'label\']} | "\n                f"similarity={similarities[index]:.4f}"\n            )\n            gallery.append((str(preview_path), caption))\n\n        rows.append(\n            [\n                rank,\n                row["label"],\n                int(row["file_id"]),\n                float(similarities[index]),\n            ]\n        )\n\n    return gallery, rows\n\n\ndef run_inference(file_obj, top_k):\n    if file_obj is None:\n        return None, {}, [], []\n\n    preview, batch = read_sar_image(file_obj)\n    predictions = predict_class(batch)\n    gallery, rows = retrieve_images(batch, top_k)\n\n    return preview, predictions, gallery, rows\n\n\nwith gr.Blocks(title=CONFIG.get("space_title", MODEL_NAME)) as demo:\n    gr.Markdown(\n        f"# {CONFIG.get(\'space_title\', MODEL_NAME)}\\n"\n        f"Model: **{MODEL_NAME}**"\n    )\n\n    with gr.Row():\n        sar_file = gr.File(\n            file_types=[".tif", ".tiff"],\n            type="binary",\n            label="Upload dual-polarization SAR TIFF",\n        )\n        top_k = gr.Slider(\n            minimum=1,\n            maximum=20,\n            value=TOP_K_DEFAULT,\n            step=1,\n            label="Number of retrieved images",\n        )\n\n    run_button = gr.Button("Classify and retrieve")\n\n    query_preview = gr.Image(label="SAR preview")\n    class_output = gr.Label(\n        num_top_classes=5,\n        label="Land-cover prediction",\n    )\n    gallery_output = gr.Gallery(\n        label="Similar SAR images",\n        columns=5,\n        object_fit="contain",\n        height="auto",\n    )\n    table_output = gr.Dataframe(\n        headers=["rank", "label", "file_id", "similarity"],\n        datatype=["number", "str", "number", "number"],\n        interactive=False,\n        label="Retrieval results",\n    )\n\n    run_button.click(\n        fn=run_inference,\n        inputs=[sar_file, top_k],\n        outputs=[\n            query_preview,\n            class_output,\n            gallery_output,\n            table_output,\n        ],\n    )\n\nif __name__ == "__main__":\n    demo.launch()\n'
REQUIREMENTS = 'tensorflow-cpu==2.16.1\ngradio==5.35.0\nnumpy==1.26.4\npandas==2.2.2\nPillow==10.4.0\ntifffile==2024.8.30\n'
README_TEXT = '---\ntitle: EuroSAT SAR Retrieval\nemoji: 📡\ncolorFrom: gray\ncolorTo: blue\nsdk: gradio\nsdk_version: 5.35.0\napp_file: app.py\npinned: false\n---\n\n# EuroSAT SAR Retrieval\n\nThis Hugging Face Space uses **DenseNet121_SAR** for EuroSAT-SAR land-cover\nclassification and same-modal SAR image retrieval.\n\nThe original two SAR channels are converted to a three-channel model input:\n\n- Channel 1: VV\n- Channel 2: VH\n- Channel 3: mean of VV and VH\n\nEach channel is percentile-stretched before inference.\n\n## Generated artifacts\n\nRun the final notebook completely. It creates:\n\n- `DenseNet121_SAR_classifier.keras`\n- `DenseNet121_SAR_embedding.keras`\n- `DenseNet121_SAR_gallery_embeddings.npy`\n- `DenseNet121_SAR_gallery_metadata.csv`\n- `gallery_previews/`\n- `config.json`\n- `app.py`\n- `requirements.txt`\n\nUpload the complete generated directory to a Hugging Face Space.\n'
CONFIG = {'space_title': 'EuroSAT SAR Retrieval', 'model_name': 'DenseNet121_SAR', 'image_size': 64, 'input_channels': 2, 'model_channels': 3, 'class_names': ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'], 'model_file': 'DenseNet121_SAR_classifier.keras', 'embedding_model_file': 'DenseNet121_SAR_embedding.keras', 'gallery_embeddings_file': 'DenseNet121_SAR_gallery_embeddings.npy', 'gallery_metadata_file': 'DenseNet121_SAR_gallery_metadata.csv', 'top_k_default': 5, 'distance': 'cosine_similarity'}

(DEPLOY_DIR / 'app.py').write_text(APP_CODE, encoding='utf-8')
(DEPLOY_DIR / 'requirements.txt').write_text(
    REQUIREMENTS,
    encoding='utf-8',
)
(DEPLOY_DIR / 'README.md').write_text(
    README_TEXT,
    encoding='utf-8',
)
with open(DEPLOY_DIR / 'config.json', 'w', encoding='utf-8') as file:
    json.dump(CONFIG, file, indent=2)


In [ ]:
required_files = [
    DEPLOY_DIR / "app.py",
    DEPLOY_DIR / "requirements.txt",
    DEPLOY_DIR / "README.md",
    DEPLOY_DIR / "config.json",
    CLASSIFIER_DEPLOY_PATH,
    EMBEDDING_DEPLOY_PATH,
    EMBEDDINGS_DEPLOY_PATH,
    METADATA_DEPLOY_PATH,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing deployment files: {missing_files}"
    )

saved_embeddings = np.load(EMBEDDINGS_DEPLOY_PATH)
saved_metadata = pd.read_csv(METADATA_DEPLOY_PATH)

assert saved_embeddings.shape[0] == len(saved_metadata)
assert saved_embeddings.shape[1] == gallery_embeddings.shape[1]
assert saved_metadata["relative_preview_path"].isna().sum() == 0
assert all(
    (DEPLOY_DIR / path).exists()
    for path in saved_metadata["relative_preview_path"]
)

archive_base = WORK_DIR / f"{MODEL_NAME}_huggingface_space"
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=DEPLOY_DIR,
)

print("SAR deployment validation passed.")
print("Space directory:", DEPLOY_DIR)
print("ZIP archive:", archive_path)
